<a href="https://colab.research.google.com/github/yunseo449/ESAA/blob/main/ESAA_YB_WEEK11_2_2_review.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# [Dacon] 제주 신용카드 빅데이터 경진대회
## 1. 데이터
- 201901-202003.csv, 202004.csv: 월별 카드 사용 데이터
- 코드에서 실제로 쓴 컬럼
  - REG_YYMM: 기준 연월
  - CARD_SIDO_NM: 카드 이용 지역(시도)
  - STD_CLSS_NM: 업종명
  - AMT: 이용 금액 (타깃)
  - CSTMR_CNT: 이용 고객 수 (변동성 지표 p를 만들 때 사용)
- submission.csv: 2020년 4월·7월의 지역×업종별 매출을 채워 넣는 제출 양식
- 목표: 2020년 7월 지역×업종별 카드 매출 예측
## 2. 코드 흐름
1. **라이브러리·데이터 로드**: pandas, numpy, sklearn(LinearRegression, PolynomialFeatures) 사용,2019.01~2020.03 데이터에 2020.04 데이터를 합침.
2. **전처리**: (지역, 업종, 연월) 단위로 AMT를 합산하고 로그를 취함. 지표가 RMSLE이기 때문이다.
3. **그룹 특성값 계산**: 650개 (지역, 업종) 그룹마다 아래 값을 구함.
    - 표준편차 std
    - 고객 수 3명 이하인 행의 비율 p (변동성 지표)
    - 선형회귀 R²
    - 4차 다항회귀의 rmse, diff_per
4. 클러스터링: 임계값 기준으로 8개 그룹으로 나눔. 그룹 크기: 13 / 126 / 71 / 1 / 6 / 14 / 363 / 56
5. 클러스터별 예측: non_cov(코로나가 없었다면 나왔을 매출) x cov_ratio(코로나 영향력)
6. 검증: 같은 방식으로 4월을 예측해 실제 4월과 비교(제주 가중치 3배, val ≈ 1.83). 그 뒤 7월을 예측
## 3. 주요 코드 분석
1. non_cov_07 x cov_ratio_07
    - non_cov_07: 코로나가 없었다고 가정한 7월 매출. 클러스터마다 다른 달들의 평균으로 구함.
    - cov_ratio_07: 코로나로 매출이 얼마나 줄었는지를 나타내는 비율. 2020년 매출 ÷ 2019년 같은 시기 매출
2. if-else 트리
- 2019.01~2020.01 데이터가 모두 있는가? → 결측이 있으면 group 8
- std ≤ 0.15 → 안정적인 그룹, group 7
- p < 0.5인가 → 변동성 작음 / 큼
- 선형회귀 R² > 0.5 → 선형적
- 다항회귀 rmse < 0.3 이고 diff_per < 30% → 계절성 있음, 아니면 규칙 없음
3. 회복 가정이 들어간 cov_ratio
    - 예 : group 3
      - r = (2020년 2·3·4월 평균) ÷ (2019년 2·3·4월 평균)
      - cov_ratio_07 = (1 + 4r) ÷ 5
        - “7월에는 4월보다 코로나 영향이 조금 회복될 것”이라는 가정을 가중평균으로 넣은 것


```python
        if (19년 데이터 + 20년 1월) 전부 존재:
            (19년 데이터 + 20년 1월)만 가지고 std를 계산
            if std > 0.15: # error_group
                if p < 0.5:  ## 변동성이 적다.

                    if 변화가 선형적이다: .........1)
                        cov_ratio_07 = (1 + 3 * mean(20Y02M, 20Y03M, 20Y04M) / mean(19Y07M, 19Y08M, 19Y09M)) / 4
                        cov_ratio_04 = mean(20Y02M, 20Y03M) / mean(19Y07M, 19Y08M, 19Y09M)
                        
                        non_cov_07 = mean(19Y11M, 19Y12M, 20Y01M)
                        non_cov_04 = mean(19Y11M, 19Y12M, 20Y01M)
                        
                        예측 7월 = non_cov_07 * cov_ratio_07
                        예측 4월 = non_cov_04 * cov_ratio_04
                    
                    else 변화가 비선형적이다:
                    
                        if 계절성을 갖는다.: .........2)
                            # 다항회귀 모델 예측후 rmse 가 0.3보다 작고, diff_per가 30% 미만일 경우
                            cov_ratio_07 = (1 + 3 * 20Y04M/19Y04M) / 4
                            cov_ratio_04 = 20Y04M/19Y04M

                            non_cov_07 = mean(19Y06M, 19Y07M, 19Y08M)
                            non_cov_04 = mean(19Y03M, 19Y04M, 19Y05M)
                            
                            예측 7월 = non_cov_07 * cov_ratio_07
                            예측 4월 = non_cov_04 * cov_ratio_04

                        elif 특별한 규칙이 없다: .........3)
                            # 다항회귀 모델 예측후 rmse 가 0.3보다 크거나 같고, diff_per가 30% 이상일 경우
                            cov_ratio_07 = (1 + 4 * mean(20Y02M, 20Y03M, 20Y04M) / mean(19Y02M, 19Y03M, 19Y04M)) / 5
                            cov_ratio_04 = mean(20Y02M, 20Y03M) / mean(19Y02M, 19Y03M)
                            
                            non_cov_07 = mean(19Y05M, 19Y06M, 19Y07M, 19Y08M)
                            non_cov_04 = mean(19Y02M, 19Y03M, 19Y04M, 19Y05M)
                            
                            예측 7월 = non_cov_07 * cov_ratio_07
                            예측 4월 = non_cov_04 * cov_ratio_04
                            
                elif p >= 0.5:  ## 변동성이 크다.
                
                    if 변화가 선형적이다: .........4)
                        cov_ratio_07 = (1 + 3 * mean(20Y02M, 20Y03M, 20Y04M) / mean(19Y07M, 19Y08M, 19Y09M)) / 4
                        cov_ratio_04 = mean(20Y02M, 20Y03M) / mean(19Y07M, 19Y08M, 19Y09M)
                        
                        non_cov_07 = mean(19Y05M, 19Y06M, 19Y07M, 19Y08M)
                        non_cov_04 = mean(19Y05M, 19Y06M, 19Y07M, 19Y08M)
                        
                        예측 7월 = non_cov_07 * cov_ratio_07
                        예측 4월 = non_cov_04 * cov_ratio_04

                    else 변화가 비선형적이다:
                        if 계절성을 갖는다.: .........5)
                            # (다항회귀 모델 예측후 rmse 가 0.3보다 작다) and (diff_per가 30% 미만이다)
                            cov_ratio_07 = (1 + 4 * mean(20Y02M, 20Y03M, 20Y04M) / mean(19Y02M, 19Y03M, 19Y04M)) / 5
                            cov_ratio_04 = mean(20Y02M, 20Y03M) / mean(19Y02M, 19Y03M)
                            
                            non_cov_07 = mean(19Y05M, 19Y06M, 19Y07M, 19Y08M, 19Y09M)
                            non_cov_04 = mean(19Y02M, 19Y03M, 19Y04M, 19Y05M, 19Y06M)
                            
                            예측 7월 = non_cov_07 * cov_ratio_07
                            예측 4월 = non_cov_04 * cov_ratio_04

                        elif 특별한 규칙이 없다: .........6)
                            # (다항회귀 모델 예측후 rmse 가 0.3보다 크다) or (diff_per가 30% 이상이다)
                            cov_ratio_07 = (1 + 4 * mean(20Y02M, 20Y03M, 20Y04M) / mean(19Y02M, 19Y03M, 19Y04M)) / 5
                            cov_ratio_04 = mean(20Y02M, 20Y03M) / mean(19Y02M, 19Y03M)
                            
                            non_cov_07 = ((19Y05M, 19Y06M, 19Y07M, 19Y08M) * 2 + 나머지 기간 AMT) / (전체 기간의 길이 + 4)
                            non_cov_04 = ((19Y02M, 19Y03M, 19Y04M, 19Y05M) * 2 + 나머지 기간 AMT) / (전체 기간의 길이 + 4)
                            
                            예측 7월 = non_cov_07 * cov_ratio_07
                            예측 4월 = non_cov_04 * cov_ratio_04

            else std <= 0.15:
                cov_ratio_07 = (1 + 3 * 20Y04M/19Y04M) / 4
                cov_ratio_04 = 20Y03M/19Y03M
                
                non_cov_07 = mean(19Y06M, 19Y07M, 19Y08M)
                non_cov_04 = mean(19Y03M, 19Y04M, 19Y06M)
                
                전체 평균 * cov_ratio .........7)
                
        else 19년 데이터가 전부 존재하지 않는다: # uncomplete_group
            # cov_ratio_07 구하기
            if 20년 4월 데이터가 존재한다:
                cov_ratio_07 = 20Y04M / 전체평균
            elif 20년 3월 데이터가 존재한다:
                cov_ratio_07 = 0.7 * 20Y03M / 전체평균
            else:
                cov_ratio_07 = 0.4
                
            # cov_ratio_04 구하기
            if 20년 3월 데이터가 존재한다:
                cov_ratio_04 = 20Y03M / 전체평균
            elif 20년 3월 데이터가 존재한다:
                cov_ratio_04 = 20Y02M / 전체평균
            else:
                cov_ratio_04 = 0.4
            
            # 결측을 0으로 채워준다.
            # 20년 1월까지 순차적으로 가중치를 크게 부여한다.
            for i, month in enumerate([20Y01M, 19Y12M, 19Y11M, ..., 19Y01M]):
                총 AMT += AMT[month] * 0.9^(i+1)
            가중치 평균 AMT = 총 AMT / (0.9 + 0.9^2 + 0.9^3 + ...)
                
            non_cov_07 = 가중치 평균 AMT
            non_cov_04 = 가중치 평균 AMT
```

## 4. 새롭게 알게 된 점 / 배운 점
- 복잡한 모델 없이도 데이터를 그룹별 특성(변동성·선형성·계절성)으로 나누고, 그룹마다 다른 가정을 적용하는 도메인 기반 접근이 효과적일 수 있다는 걸 알았다.
- 예측 대상을 '평상시 수준 x 외부 충격 비율'로 분해하는 아이디어는 코로나처럼 구조적 변화가 있는 시계열에서 해석력이 높았다.
- 저자가 아쉬운 점으로 시계열 분해(추세·계절성·주기)를 더 일찍 공부하지 못한 걸 꼽았는데, STL 분해 같은 방법을 적용해 보면 좋겠다.
- 한계: 임계값을 수동으로 정해 근거가 약하고, 4월로 한 검증이 7월 상황을 충분히 대변하지 못할 수 있다.